# Verifikasi kandidat SENA v0.7 dan regresi NARA

Ekspor setelah pelatihan selesai. Kandidat terikat hash bobot, konfigurasi v0.4, dan lock v0.7. Target dan cutoff 0.95 tetap. Tidak ada parameter fitting dalam notebook verifikasi ini.

Development manusia146 serta asisten74 yang sudah dilihat dipakai memilih model. Regresi manusia50 lama bukan penerimaan independen. Diagnostik50 baru v0.7 dibekukan sebelum pelatihan, diperiksa sesudah pemilihan, dan tidak dipakai mengubah kandidat. Label diagnostik adalah usulan asisten, belum acuan manusia. Persetujuan mutu dan Done tetap memerlukan proses penerimaan manusia.

In [ ]:
from pathlib import Path
import os,sys,json,subprocess
ROOT=Path(os.environ['SLR09_PACKAGE_ROOT'])
WORK=Path(os.environ['SLR09_WORK_DIR'])
RUN=WORK/'training_matched_v07'
OUTPUT=WORK/'quality_diagnostics_v07'
def execute(command):
 p=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
 for line in p.stdout: print(line,end='',flush=True)
 p.wait()
 if p.returncode: raise RuntimeError('Step failed: '+str(p.returncode))
assert json.loads((RUN/'run_state.json').read_text())['status']=='SUCCEEDED'

In [ ]:
execute([sys.executable,str(ROOT/'code/quality_fix/prepare_matched_candidate_v07.py'),
 '--run',str(RUN),'--output',str(WORK/'exports')])

In [ ]:
execute([sys.executable,str(ROOT/'code/run_active_tests.py'),
 '--evidence-dir',str(WORK/'contract_tests_v07')])

In [ ]:
execute([sys.executable,str(ROOT/'code/quality_fix/run_matched_diagnostics_v07.py'),
 '--previous',str(ROOT/'evidence/quality_fix_v05/previous_50'),
 '--seen',str(ROOT/'data/mvp_seen_development_v07.json'),
 '--fresh',str(ROOT/'data/frozen_new_diagnostic_v0.7.json'),
 '--sena-dir',str(RUN/'best_model'),
 '--nara-archive',str(WORK/'assets/CT/SILAR_NARA_reviewed_pilot_20261018T001122_607337Z.zip'),
 '--training-run',str(RUN),'--output',str(OUTPUT)])

In [ ]:
state=json.loads((OUTPUT/'run_state.json').read_text())
assert state['status']=='SUCCEEDED' and state['model_inference_executed']
assert not state['parameter_fitting_executed'] and not state['quality_acceptance_approved']
print(json.dumps(state['diagnostic_summary'],indent=2))
print('Human acceptance remains pending; no production deployment.')